In [1]:
%pip install -q catboost pandas numpy scikit-learn 


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import catboost
import numpy
import pandas

print("catboost:", catboost.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)

catboost: 1.2.8
numpy: 1.26.4
pandas: 2.3.3


In [3]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42
TARGET = "target"
MODEL_PATH = Path("../fraud_detector/models/model.cbm")
META_PATH = Path("../fraud_detector/models/model_meta.json")
TRAIN_PATH = Path("../data/train.csv")

In [4]:
train_raw = pd.read_csv(TRAIN_PATH)

print("Shape:", train_raw.shape)

display(train_raw.head())

display(
    train_raw[TARGET]
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
    .assign(share=lambda x: x["count"] / x["count"].sum())
)

Shape: (786431, 18)


,transaction_time,merch,cat_id,amount,name_1,name_2,gender,street,one_city,us_state,post_code,lat,lon,population_city,jobs,merchant_lat,merchant_lon,target
0,2019-12-27 15:21,fraud_Cormier LLC,health_fitness,148.04,Daniel,Martinez,M,8510 Acevedo Burgs,Kent,OR,97033,45.0838,-120.6649,60,Museum education officer,45.042827,-120.709327,0
1,2019-04-17 23:09,"fraud_Brown, Homenick and Lesch",health_fitness,39.40,Grace,Williams,F,28812 Charles Mill Apt. 628,Plantersville,AL,36758,32.6176,-86.9475,1412,Drilling engineer,31.872266,-87.828247,0
2,2019-09-23 15:02,fraud_Ruecker-Mayert,kids_pets,52.96,Kyle,Park,M,7507 Larry Passage Suite 859,Mount Perry,OH,43760,39.8788,-82.1880,1831,Barrister's clerk,40.010874,-81.841249,0
3,2019-05-13 16:00,"fraud_Mante, Luettgen and Hackett",health_fitness,7.66,Monique,Martin,F,68276 Matthew Springs,Ratcliff,TX,75858,31.3833,-95.0619,43,"Engineer, production",30.888406,-95.141609,0
4,2019-08-18 07:27,fraud_Luettgen PLC,gas_transport,51.59,Christine,Johnson,F,8011 Chapman Tunnel Apt. 568,Blairsden-Graeagle,CA,96103,39.8127,-120.6405,1725,Chartered legal executive (England and Wales),39.376017,-121.311691,0


,count,share
target,,
0,781927,0.994273
1,4504,0.005727


In [5]:
def haversine(lat1, lon1, lat2, lon2):
    rad = 6371

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2)
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return rad * c

In [ ]:
def add_features(df):
    df_ = df.copy()

    coordinate_columns = [
        "lat",
        "lon",
        "merchant_lat",
        "merchant_lon",
    ]

    for col in coordinate_columns:
        df_[col] = pd.to_numeric(df_[col], errors="coerce")

    df_["haversine"] = haversine(
        df_["lat"],
        df_["lon"],
        df_["merchant_lat"],
        df_["merchant_lon"],
    )

    df_["transaction_time"] = pd.to_datetime(df_["transaction_time"], errors="coerce")
    df_["year"] = df_["transaction_time"].dt.year
    df_["month"] = df_["transaction_time"].dt.month
    df_["day"] = df_["transaction_time"].dt.day
    df_["hour"] = df_["transaction_time"].dt.hour
    df_["minute"] = df_["transaction_time"].dt.minute
    df_["weekend"] = (df_["transaction_time"].dt.dayofweek > 4).astype("int")

    columns_to_drop = ["transaction_time", "name_1", "name_2", "street"]

    return df_.drop(columns=columns_to_drop, errors="ignore")



def run_preproc(input_df):
    return add_features(input_df)

In [7]:
train = run_preproc(train_raw)

y = train[TARGET].astype(int)
X = train.drop(columns=[TARGET])

print("Processed shape:", X.shape)
display(X.head())

Processed shape: (786431, 20)


,merch,cat_id,amount,gender,one_city,us_state,post_code,lat,lon,population_city,jobs,merchant_lat,merchant_lon,haversine,year,month,day,hour,minute,weekend
0,fraud_Cormier LLC,health_fitness,148.04,M,Kent,OR,97033,45.0838,-120.6649,60,Museum education officer,45.042827,-120.709327,5.738654,2019,12,27,15,21,0
1,"fraud_Brown, Homenick and Lesch",health_fitness,39.40,F,Plantersville,AL,36758,32.6176,-86.9475,1412,Drilling engineer,31.872266,-87.828247,117.172182,2019,4,17,23,9,0
2,fraud_Ruecker-Mayert,kids_pets,52.96,M,Mount Perry,OH,43760,39.8788,-82.1880,1831,Barrister's clerk,40.010874,-81.841249,33.007268,2019,9,23,15,2,0
3,"fraud_Mante, Luettgen and Hackett",health_fitness,7.66,F,Ratcliff,TX,75858,31.3833,-95.0619,43,"Engineer, production",30.888406,-95.141609,55.550169,2019,5,13,16,0,0
4,fraud_Luettgen PLC,gas_transport,51.59,F,Blairsden-Graeagle,CA,96103,39.8127,-120.6405,1725,Chartered legal executive (England and Wales),39.376017,-121.311691,75.267261,2019,8,18,7,27,1


In [8]:
categorical_candidates = [
    "merch",
    "cat_id",
    "gender",
    "one_city",
    "us_state",
    "post_code",
]

categorical_columns = [col for col in categorical_candidates if col in X.columns]

auto_categorical = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

categorical_columns = sorted(set(categorical_columns + auto_categorical))

for col in categorical_columns:
    X[col] = X[col].fillna("__MISSING__").astype(str)

print("Categorical columns:")
print(categorical_columns)

print("\nNumeric columns:")
print([c for c in X.columns if c not in categorical_columns])

Categorical columns:
['cat_id', 'gender', 'jobs', 'merch', 'one_city', 'post_code', 'us_state']

Numeric columns:
['amount', 'lat', 'lon', 'population_city', 'merchant_lat', 'merchant_lon', 'haversine', 'year', 'month', 'day', 'hour', 'minute', 'weekend']


In [9]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)

print("\nTrain target share:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nValid target share:")
print(y_valid.value_counts(normalize=True).sort_index())

Train: (629144, 20)
Valid: (157287, 20)

Train target share:
target
0    0.994273
1    0.005727
Name: proportion, dtype: float64

Valid target share:
target
0    0.994272
1    0.005728
Name: proportion, dtype: float64


In [10]:
model = CatBoostClassifier(
    iterations=200,
    depth=7,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    task_type="CPU",
    thread_count=-1,
    verbose=50,
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_columns,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
    early_stopping_rounds=75,
)

0:	test: 0.9744045	best: 0.9744045 (0)	total: 366ms	remaining: 1m 12s
50:	test: 0.9939687	best: 0.9939687 (50)	total: 20.5s	remaining: 59.9s
100:	test: 0.9959752	best: 0.9959752 (100)	total: 37s	remaining: 36.3s
150:	test: 0.9967043	best: 0.9967043 (150)	total: 51.9s	remaining: 16.8s
199:	test: 0.9973428	best: 0.9973428 (199)	total: 1m 6s	remaining: 0us

bestTest = 0.9973427825
bestIteration = 199



In [11]:
valid_score = model.predict_proba(X_valid)[:, 1]

roc_auc = roc_auc_score(y_valid, valid_score)
pr_auc = average_precision_score(y_valid, valid_score)

print(f"ROC-AUC: {roc_auc:.5f}")
print(f"PR-AUC:  {pr_auc:.5f}")

pd.Series(valid_score).describe()

ROC-AUC: 0.99734
PR-AUC:  0.85682


count    157287.000000
mean          0.051375
std           0.135485
min           0.000040
25%           0.003485
50%           0.009221
75%           0.030905
max           0.999998
dtype: float64

In [12]:
model.save_model(MODEL_PATH)

model_meta = {
    "target": TARGET,
    "feature_names": X.columns.tolist(),
    "categorical_features": categorical_columns,
}

with META_PATH.open("w", encoding="utf-8") as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=2)

print("Saved model:", MODEL_PATH.resolve())
print("Saved metadata:", META_PATH.resolve())
print("Model size, MB:", round(MODEL_PATH.stat().st_size / 1024**2, 2))

Saved model: /Users/stasya/Yandex.Disk.localized/4 курс/MLOps/MLOps-hw1/fraud_detector/models/model.cbm
Saved metadata: /Users/stasya/Yandex.Disk.localized/4 курс/MLOps/MLOps-hw1/fraud_detector/models/model_meta.json
Model size, MB: 65.78
